In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/10 04:38:19 WARN Utils: Your hostname, codespaces-e8ab3a, resolves to a loopback address: 127.0.0.1; using 10.0.2.47 instead (on interface eth0)
26/03/10 04:38:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 04:38:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-01.parquet

--2026-03-10 04:40:19--  https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-01.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.35.33.83, 13.35.33.60, 13.35.33.10, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.35.33.83|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 308924937 (295M) [application/x-www-form-urlencoded]
Saving to: ‘fhvhv_tripdata_2021-01.parquet’

fhvhv_tripdata_2021 100%[===================>] 294.61M  17.9MB/s    in 18s     

2026-03-10 04:40:38 (16.0 MB/s) - ‘fhvhv_tripdata_2021-01.parquet’ saved [308924937/308924937]



In [13]:
ls

README.md                       main.py         pyspark.ipynb    uv.lock
fhvhv_tripdata_2021-01.parquet  pyproject.toml  test_pyspark.py


In [9]:
# df = pd.read_parquet('fhvhv_tripdata_2021-01.parquet')
df = spark.read \
    .option("header","true") \
    .parquet("fhvhv_tripdata_2021-01.parquet")

In [11]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('originating_base_num', StringType(), True), StructField('request_datetime', TimestampNTZType(), True), StructField('on_scene_datetime', TimestampNTZType(), True), StructField('pickup_datetime', TimestampNTZType(), True), StructField('dropoff_datetime', TimestampNTZType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('trip_miles', DoubleType(), True), StructField('trip_time', LongType(), True), StructField('base_passenger_fare', DoubleType(), True), StructField('tolls', DoubleType(), True), StructField('bcf', DoubleType(), True), StructField('sales_tax', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('airport_fee', DoubleType(), True), StructField('tips', DoubleType(), True), StructField('driver_pay', DoubleType(), True), StructField('shared_re

In [12]:
df.head()

Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', originating_base_num='B02682', request_datetime=datetime.datetime(2021, 1, 1, 0, 28, 9), on_scene_datetime=datetime.datetime(2021, 1, 1, 0, 31, 42), pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, trip_miles=5.26, trip_time=923, base_passenger_fare=22.28, tolls=0.0, bcf=0.67, sales_tax=1.98, congestion_surcharge=2.75, airport_fee=None, tips=0.0, driver_pay=14.99, shared_request_flag='N', shared_match_flag='N', access_a_ride_flag=' ', wav_request_flag='N', wav_match_flag='N')

In [29]:
type(df)

pyspark.sql.classic.dataframe.DataFrame

In [30]:
df.limit(101).coalesce(1).write.option("header", True).csv("sample")

In [24]:
import pandas as pd

In [31]:
df_pandas = pd.read_csv('sample/part-00000-63f7f53e-ba92-4191-b678-e19e672dc9c8-c000.csv')

In [34]:
df_pandas.dtypes

hvfhs_license_num           str
dispatching_base_num        str
originating_base_num        str
request_datetime            str
on_scene_datetime           str
pickup_datetime             str
dropoff_datetime            str
PULocationID              int64
DOLocationID              int64
trip_miles              float64
trip_time                 int64
base_passenger_fare     float64
tolls                   float64
bcf                     float64
sales_tax               float64
congestion_surcharge    float64
airport_fee             float64
tips                    float64
driver_pay              float64
shared_request_flag         str
shared_match_flag           str
access_a_ride_flag          str
wav_request_flag            str
wav_match_flag              str
dtype: object

In [36]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('originating_base_num', StringType(), True), StructField('request_datetime', StringType(), True), StructField('on_scene_datetime', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('trip_miles', DoubleType(), True), StructField('trip_time', LongType(), True), StructField('base_passenger_fare', DoubleType(), True), StructField('tolls', DoubleType(), True), StructField('bcf', DoubleType(), True), StructField('sales_tax', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('airport_fee', DoubleType(), True), StructField('tips', DoubleType(), True), StructField('driver_pay', DoubleType(), True), StructField('shared_request_flag', StringType(

In [39]:
from pyspark.sql import types

In [67]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True), 
    types.StructField('dispatching_base_num', types.StringType(), True), 
    types.StructField('originating_base_num', types.StringType(), True), 
    types.StructField('request_datetime', types.TimestampType(), True), 
    types.StructField('on_scene_datetime', types.TimestampType(), True), 
    types.StructField('pickup_datetime', types.TimestampType(), True), 
    types.StructField('dropoff_datetime', types.TimestampType(), True), 
    types.StructField('PULocationID', types.LongType(), True),
    types.StructField('DOLocationID', types.LongType(), True),
    types.StructField('trip_miles', types.DoubleType(), True),
    types.StructField('trip_time', types.LongType(), True), 
    types.StructField('base_passenger_fare', types.DoubleType(), True), 
    types.StructField('tolls', types.DoubleType(), True), 
    types.StructField('bcf', types.DoubleType(), True), 
    types.StructField('sales_tax', types.DoubleType(), True), 
    types.StructField('congestion_surcharge', types.DoubleType(), True), 
    types.StructField('airport_fee', types.DoubleType(), True), 
    types.StructField('tips', types.DoubleType(), True), 
    types.StructField('driver_pay', types.DoubleType(), True), 
    types.StructField('shared_request_flag', types.StringType(), True), 
    types.StructField('shared_match_flag', types.StringType(), True), 
    types.StructField('access_a_ride_flag', types.StringType(), True), 
    types.StructField('wav_request_flag', types.StringType(), True), 
    types.StructField('wav_match_flag', types.StringType(), True)]
    )

In [68]:
df = spark.read \
    .option("header","true") \
    .schema(schema) \
    .parquet("fhvhv_tripdata_2021-01.parquet")

In [69]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('originating_base_num', StringType(), True), StructField('request_datetime', TimestampType(), True), StructField('on_scene_datetime', TimestampType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('trip_miles', DoubleType(), True), StructField('trip_time', LongType(), True), StructField('base_passenger_fare', DoubleType(), True), StructField('tolls', DoubleType(), True), StructField('bcf', DoubleType(), True), StructField('sales_tax', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('airport_fee', DoubleType(), True), StructField('tips', DoubleType(), True), StructField('driver_pay', DoubleType(), True), StructField('shared_request_flag',

In [70]:
df.repartition(20)

DataFrame[hvfhs_license_num: string, dispatching_base_num: string, originating_base_num: string, request_datetime: timestamp, on_scene_datetime: timestamp, pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: bigint, DOLocationID: bigint, trip_miles: double, trip_time: bigint, base_passenger_fare: double, tolls: double, bcf: double, sales_tax: double, congestion_surcharge: double, airport_fee: double, tips: double, driver_pay: double, shared_request_flag: string, shared_match_flag: string, access_a_ride_flag: string, wav_request_flag: string, wav_match_flag: string]

In [72]:
df.write.parquet('fhvhv/2021/01/')

In [81]:
df = spark.read.parquet('fhvhv/2021/01/')

In [83]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp (nullable = true)
 |-- on_scene_datetime: timestamp (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_ride_flag: string (nul

In [88]:
df \
    .select('hvfhs_license_num','request_datetime','pickup_datetime','dropoff_datetime','trip_miles','trip_time') \
    .filter(df.hvfhs_license_num == 'HV0003') \
    .show()

[Stage 31:>                                                         (0 + 1) / 1]

+-----------------+-------------------+-------------------+-------------------+----------+---------+
|hvfhs_license_num|   request_datetime|    pickup_datetime|   dropoff_datetime|trip_miles|trip_time|
+-----------------+-------------------+-------------------+-------------------+----------+---------+
|           HV0003|2021-01-01 00:28:09|2021-01-01 00:33:44|2021-01-01 00:49:07|      5.26|      923|
|           HV0003|2021-01-01 00:45:56|2021-01-01 00:55:19|2021-01-01 01:18:21|      3.65|     1382|
|           HV0003|2021-01-01 00:21:15|2021-01-01 00:23:56|2021-01-01 00:38:05|      3.51|      849|
|           HV0003|2021-01-01 00:39:12|2021-01-01 00:42:51|2021-01-01 00:45:50|      0.74|      179|
|           HV0003|2021-01-01 00:46:11|2021-01-01 00:48:14|2021-01-01 01:08:42|       9.2|     1228|
|           HV0003|2021-01-01 00:10:36|2021-01-01 00:14:30|2021-01-01 00:50:27|     13.53|     2157|
|           HV0003|2021-01-01 00:21:17|2021-01-01 00:22:54|2021-01-01 00:30:20|       1.6| 

In [89]:
from pyspark.sql import functions as F

In [90]:
F.to_date(df.pickup_datetime)

Column<'to_date(pickup_datetime)'>

In [95]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .select('hvfhs_license_num','request_datetime','pickup_datetime','dropoff_datetime','trip_miles','trip_time','pickup_date','dropoff_date') \
    .show()
    

+-----------------+-------------------+-------------------+-------------------+----------+---------+-----------+------------+
|hvfhs_license_num|   request_datetime|    pickup_datetime|   dropoff_datetime|trip_miles|trip_time|pickup_date|dropoff_date|
+-----------------+-------------------+-------------------+-------------------+----------+---------+-----------+------------+
|           HV0003|2021-01-01 00:28:09|2021-01-01 00:33:44|2021-01-01 00:49:07|      5.26|      923| 2021-01-01|  2021-01-01|
|           HV0003|2021-01-01 00:45:56|2021-01-01 00:55:19|2021-01-01 01:18:21|      3.65|     1382| 2021-01-01|  2021-01-01|
|           HV0003|2021-01-01 00:21:15|2021-01-01 00:23:56|2021-01-01 00:38:05|      3.51|      849| 2021-01-01|  2021-01-01|
|           HV0003|2021-01-01 00:39:12|2021-01-01 00:42:51|2021-01-01 00:45:50|      0.74|      179| 2021-01-01|  2021-01-01|
|           HV0003|2021-01-01 00:46:11|2021-01-01 00:48:14|2021-01-01 01:08:42|       9.2|     1228| 2021-01-01|  2021